# SwarmSort Phase 5b: full training of the Arm C winner (PSO)

Phase 4 searched the six training hyperparameters with cheap proxy trainings (12 epochs on 40% of the train split).
This notebook fully trains the best configuration found by the particle swarm optimizer (Arm C, 8 particles x 4 rounds, `phase4a_pso_search.ipynb`):
`results/best_pso.json`, proxy val mAP@50 0.4953 (evaluation r4_p3).

It uses exactly the Arm A recipe: 100 epochs, imgsz 416, batch 16, seed 42, deterministic, SGD, warmup 3 epochs, mosaic
closed for the last 10, one GPU. All of this is fixed in `src/training.py`, so only the six searched values below differ
from the Arm A baseline and the comparison is fair. Validation on the val split runs every epoch; the reported numbers are
the val metrics of `best.pt`. The test split is not used here: it is evaluated exactly once, locally, after both Phase 5
trainings are done.

| Hyperparameter | Arm C (PSO, trained here) | Arm A (Ultralytics defaults) |
|---|---|---|
| lr0 | 0.00315345 | 0.01 |
| lrf | 0.0751247 | 0.01 |
| momentum | 0.949322 | 0.937 |
| weight_decay | 0.000259703 | 0.0005 |
| mosaic | 0.271828 | 1.0 |
| scale | 0.470742 | 0.5 |

**Expected time:** about 1.2 hours on GPU T4 x2 (the Arm A training took 71 minutes).

Outputs: `/kaggle/working/SwarmSort/results/arm_c.json`, `results/arm_c_epochs.csv` and the run folder
`results/runs/arm_c/`; the summary cell also copies the weights to `/kaggle/working/arm_c_best.pt`.

**Settings:** the same as in every earlier phase: Accelerator = **GPU T4 x2** (keep it, so all results come from the same
hardware), Internet = on (for the clone, pip and the `yolov8n.pt` download), Input = the dataset
`viswaprakash1990/garbage-detection` (*Garbage Detection - 6 Waste Categories*). Run it with
**Save Version -> Save & Run All (Commit)** so it keeps running with the browser closed (up to 12 hours).

Phase 5b and Phase 5a (`phase5a_train_arm_b.ipynb`) are independent: they can run at the same time in separate
Kaggle sessions. If Kaggle queues the second one, just run them one after the other.

In [ ]:
import os
REPO = "/kaggle/working/SwarmSort"
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/Shreyansh303/SwarmSort.git {REPO}
%cd {REPO}
# Kaggle's own CUDA build of torch is kept: only Ultralytics is installed, at the version pinned in requirements.txt
!pip install -q ultralytics==8.4.169
import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "|", torch.cuda.get_device_name(0))

In [ ]:
# Rebuild configs/data.yaml and configs/lists/ with Kaggle paths from the committed configs/split.csv
!python src/verify_dataset.py --root /kaggle/input
assert os.path.exists("configs/data.yaml"), "verify_dataset.py did not write configs/data.yaml"
print(open("configs/data.yaml").read())

In [ ]:
# Full training of the Arm C winner on one GPU (device 0), exactly as Arm A was trained.
# check=True: if train_final.py stops with an error, this cell fails and the version is marked failed
import os, subprocess, sys
subprocess.run([sys.executable, "src/train_final.py", "--config", "results/best_pso.json", "--name", "arm_c",
                "--out", "results/arm_c.json", "--device", "0"], check=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})


In [ ]:
# Arm C next to Arm A (val split, best.pt), a check that the intended config was trained,
# and a copy of the weights at the top of /kaggle/working so they are easy to find in the Output tab
import json, math, os, shutil, sys
sys.path.insert(0, "src")
from search_space import NAMES
ARM, LETTER, CONFIG = "arm_c", "C", "results/best_pso.json"
run = json.load(open(f"results/{ARM}.json"))
base = json.load(open("results/baseline.json"))  # Arm A, committed in Phase 2
wanted = json.load(open(CONFIG))["params"]

wrong = [k for k in NAMES if not math.isclose(run["params"][k], wanted[k], rel_tol=1e-9)]
assert not wrong, f"results/{ARM}.json was trained with other values than {CONFIG} for {wrong}"
assert run["epochs"] == 100 and run["imgsz"] == 416 and not run["smoke"], "not the full 100-epoch Arm A recipe"
print(f"Trained config matches {CONFIG}:", ", ".join(f"{k}={run['params'][k]:g}" for k in NAMES))

print(f"\n{'val split, best.pt':<20}{'Arm ' + LETTER:>12}{'Arm A':>12}{'difference':>12}")
for key, label in [("map50", "mAP@50"), ("map50_95", "mAP@50-95"), ("precision", "precision"), ("recall", "recall")]:
    print(f"{label:<20}{run[key]:>12.4f}{base[key]:>12.4f}{run[key] - base[key]:>+12.4f}")
print(f"{'training time (h)':<20}{run['seconds'] / 3600:>12.2f}{base['seconds'] / 3600:>12.2f}")
print(f"{'GPU':<20}{run['device']:>12}{base['device']:>12}")
print("Val numbers only: the final, fair comparison is the test split, evaluated once locally after Phase 5.")

weights = f"results/runs/{ARM}/weights/best.pt"
assert os.path.exists(weights), f"{weights} not found (see the output of the training cell)"
shutil.copyfile(weights, f"/kaggle/working/{ARM}_best.pt")
print(f"\nCopied {weights} -> /kaggle/working/{ARM}_best.pt ({os.path.getsize(weights) / 1e6:.1f} MB)")


### Files to download

From the **Output** tab of the finished version, download these three files and put them in the local project:

| File in the Output tab | Where it goes locally |
|---|---|
| `SwarmSort/results/arm_c.json` | `results/arm_c.json` |
| `SwarmSort/results/arm_c_epochs.csv` | `results/arm_c_epochs.csv` |
| `arm_c_best.pt` (top level of the output) | `results/runs/arm_c/weights/best.pt` (create the folders) |

**The `.pt` file downloads as a `.zip`**, because a PyTorch checkpoint is itself a zip archive. Do not unzip it: just
rename the downloaded file to `best.pt` and put it in `results/runs/arm_c/weights/`.

`arm_c.json` and `arm_c_epochs.csv` are small and get committed; the weights stay local (`results/runs/` and `*.pt` are
git-ignored). When the files of both Phase 5a and 5b are in place, run the final test-set evaluation locally (see the
Phase 5 section of the README).

### If the session dies

This is one training (about 1.2 hours), well inside Kaggle's 12-hour limit, so a dead session is simply rerun with the
same settings: the run is seeded and deterministic, so the rerun trains the same model. Nothing needs to be attached or
resumed; download the files from the version that finishes.